# LeetCode 3614 — Process String with Special Operations II bằng Python

- Link: https://leetcode.com/problems/process-string-with-special-operations-ii/
- Độ khó: **Hard**
- Chủ đề: **Truy vết ngược chỉ số** (reverse index tracking), chỉ lưu **độ dài** thay vì chuỗi, mô hình "rope" lười

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def processStr(self, s: str, k: int) -> str:
```

## 1. Mục tiêu học

1. Nhận ra chuỗi kết quả có thể dài tới `10^{15}` ⇒ **không được** dựng chuỗi; chỉ theo dõi **độ dài** sau mỗi thao tác.
2. Hỏi một ký tự ở vị trí `k` ⇒ đi **ngược** các thao tác, biến đổi `k` về vị trí tương ứng trước thao tác đó.
3. Mỗi thao tác có ánh xạ ngược đơn giản: nhân đôi (`#`) ⇒ `k mod len_trước`; đảo (`%`) ⇒ `len − 1 − k`; xoá cuối (`*`) ⇒ `k` không đổi; thêm chữ ⇒ nếu `k` là vị trí vừa thêm thì trả chữ đó.
4. Một cách nhìn khác: biểu diễn chuỗi bằng **cây thao tác lười** (rope) và đi xuống theo chỉ số.
5. Kiểm chứng bằng mô phỏng trực tiếp với chuỗi nhỏ.

## 2. Đề bài & ví dụ

Xử lý `s` từ trái sang phải: chữ thường ⇒ nối vào `result`; `*` ⇒ xoá ký tự cuối (nếu có); `#` ⇒ `result = result + result`; `%` ⇒ đảo `result`. Trả về ký tự thứ `k` (0-indexed) của `result` cuối, hoặc `'.'` nếu ngoài phạm vi.

**Ví dụ 1:** `s = "a#b%*", k = 1` → `"a"` (`result = "ba"`).

**Ví dụ 2:** `s = "cd%#*#", k = 3` → `"d"` (`result = "dcddcd"`).

**Ví dụ 3:** `s = "z*#", k = 0` → `"."`.

**Ràng buộc:** `|s| ≤ 10^5`, `k ≤ 10^{15}`, độ dài kết quả `≤ 10^{15}`.

In [1]:
import random, time

examples = [("a#b%*", 1, "a"), ("cd%#*#", 3, "d"), ("z*#", 0, ".")]
for s, k, exp in examples:
    print(repr(s), k, "-> kỳ vọng", repr(exp))

'a#b%*' 1 -> kỳ vọng 'a'
'cd%#*#' 3 -> kỳ vọng 'd'
'z*#' 0 -> kỳ vọng '.'


## 3. Kiến thức nền

### 3.1. Độ dài sau từng bước

`L[0] = 0`; `L[i+1]` = độ dài sau khi xử lý `s[i]`: chữ ⇒ `+1`; `*` ⇒ `max(0, L − 1)`; `#` ⇒ `2L`; `%` ⇒ không đổi.

In [2]:
def lengths(s):
    L = [0]
    for ch in s:
        m = L[-1]
        if ch == "*": m = max(0, m - 1)
        elif ch == "#": m *= 2
        elif ch == "%": pass
        else: m += 1
        L.append(m)
    return L

print(lengths("cd%#*#"))

[0, 1, 2, 2, 4, 3, 6]


### 3.2. Ánh xạ ngược chỉ số

Giả sử biết `k` là chỉ số trong chuỗi **sau** bước `i` (độ dài `L[i+1]`), chuỗi trước bước đó có độ dài `p = L[i]`:

- chữ `c`: chuỗi sau = trước + `c`. Nếu `k = p` ⇒ đáp án là `c`; ngược lại `k < p` giữ nguyên.
- `*`: chuỗi sau = tiền tố của chuỗi trước ⇒ `k` giữ nguyên.
- `#`: chuỗi sau = trước + trước ⇒ `k ← k − p` nếu `k ≥ p`.
- `%`: chuỗi sau = đảo của trước ⇒ `k ← p − 1 − k`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ta không cần biết **toàn bộ** chuỗi, chỉ cần biết ký tự ở **một** vị trí. Mỗi thao tác "sao chép" ký tự từ chuỗi trước sang chuỗi sau theo một ánh xạ chỉ số đơn giản; đi ngược từ cuối, ta luôn biết ký tự cần tìm nằm ở đâu trong chuỗi trước. Đến khi gặp chữ cái vừa được thêm đúng ở vị trí đó ⇒ xong.

Ví dụ 2 (`"cd%#*#"`, `k = 3`): độ dài `[0,1,2,2,4,3,6]`. Ngược: `#` (p = 3): `k = 3 ≥ 3` ⇒ `k = 0`; `*` ⇒ giữ; `#` (p = 2): `k = 0`; `%` (p = 2): `k = 1`; `d` (p = 1): `k == p` ⇒ `'d'`. ✔

## 5. Cách 1 — Brute Force: mô phỏng chuỗi

In [3]:
def brute(s, k):
    r = []
    for ch in s:
        if ch == "*":
            if r: r.pop()
        elif ch == "#": r = r + r
        elif ch == "%": r.reverse()
        else: r.append(ch)
    return r[k] if 0 <= k < len(r) else "."

for s, k, exp in examples:
    print(repr(brute(s, k)), "kỳ vọng", repr(exp))

'a' kỳ vọng 'a'
'd' kỳ vọng 'd'
'.' kỳ vọng '.'


**Nút thắt:** độ dài có thể tới `10^{15}` (chỉ cần ~50 lần `#`).

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính `L` (mục 3.1). Nếu `k ≥ L[n]` ⇒ `'.'`.
2. Duyệt `i = n−1 … 0` với `p = L[i]`, áp dụng ánh xạ ngược 3.2; trả về chữ cái khi `k == p` ở bước thêm chữ.

`O(n)` thời gian, `O(n)` bộ nhớ (có thể `O(1)` nếu tính ngược độ dài — nhưng `*` với `max(0, ·)` không khả nghịch, nên lưu mảng là an toàn).

Cũng lưu ý: độ dài tối đa `10^{15}` vừa với số nguyên Python; ở C++/Java cần `long long` và có thể chặn độ dài ở `k + 1` **chỉ khi** không còn `%` phía sau (xem Cách 4 vì sao chặn bừa là sai).

## 7. Các cách tiếp theo

- **Cách 2:** truy vết ngược với mảng độ dài (lời giải nộp).
- **Cách 3:** **rope lười**: mỗi bước tạo một nút mô tả chuỗi (`Leaf`, `Append`, `Pop`, `Double`, `Reverse`) trỏ tới nút trước; truy vấn chỉ số bằng cách đi xuống.
- **Cách 4:** mô phỏng có **cắt ngắn**: khi độ dài vượt `k + 1` mà sau đó không còn `%` hay `*` thì có thể dừng nhân đôi — chỉ đúng khi biết trước, minh hoạ vì sao đảo ngược phá mọi "cắt ngắn" đơn giản.

### 7.1. Cách 2 — Truy vết ngược (lời giải chính)

In [4]:
def reverse_track(s, k):
    L = lengths(s)
    if k >= L[-1]: return "."
    for i in range(len(s) - 1, -1, -1):
        ch = s[i]; p = L[i]
        if ch == "#":
            if k >= p: k -= p
        elif ch == "%":
            k = p - 1 - k
        elif ch == "*":
            pass
        else:
            if k == p: return ch
    return "."

for s, k, exp in examples:
    print(repr(reverse_track(s, k)), "kỳ vọng", repr(exp))

'a' kỳ vọng 'a'
'd' kỳ vọng 'd'
'.' kỳ vọng '.'


### 7.2. Cách 3 — Rope lười

In [5]:
def rope(s, k):
    # nút: (loại, độ dài, nút con, dữ liệu)
    nodes = [("empty", 0, None, None)]
    for ch in s:
        prev = len(nodes) - 1; m = nodes[prev][1]
        if ch == "*": nodes.append(("pop", max(0, m - 1), prev, None))
        elif ch == "#": nodes.append(("double", 2 * m, prev, None))
        elif ch == "%": nodes.append(("rev", m, prev, None))
        else: nodes.append(("push", m + 1, prev, ch))
    u = len(nodes) - 1
    if k >= nodes[u][1]: return "."
    while True:
        kind, m, child, data = nodes[u]
        if kind == "push":
            if k == m - 1: return data
            u = child
        elif kind == "pop": u = child
        elif kind == "double":
            half = nodes[child][1]; k = k - half if k >= half else k; u = child
        elif kind == "rev":
            k = m - 1 - k; u = child

for s, k, exp in examples:
    print(repr(rope(s, k)), "kỳ vọng", repr(exp))

'a' kỳ vọng 'a'
'd' kỳ vọng 'd'
'.' kỳ vọng '.'


**Ghi chú:** đây chính là Cách 2 viết dưới dạng cấu trúc dữ liệu — ưu điểm là có thể trả lời **nhiều** truy vấn `k` mà không cần tính lại, và mở rộng cho thao tác lồng nhau (ví dụ "nối với một phiên bản cũ").

### 7.3. Cách 4 — Vì sao không cắt ngắn được

In [6]:
def truncated_sim(s, k, cap):
    r = []
    for ch in s:
        if ch == "*":
            if r: r.pop()
        elif ch == "#": r = (r + r)[:cap]                 # chỉ giữ cap ký tự đầu
        elif ch == "%": r.reverse()
        else: r.append(ch)
    return r[k] if 0 <= k < len(r) else "."

s, k = "abc#%", 0
print("cắt ở 2 ký tự:", truncated_sim(s, k, 2), "| đúng:", brute(s, k))
assert truncated_sim(s, k, 2) != brute(s, k)
print("lý do: sau khi cắt, phép đảo đưa phần đã bị bỏ đi lên đầu chuỗi")

cắt ở 2 ký tự: b | đúng: c
lý do: sau khi cắt, phép đảo đưa phần đã bị bỏ đi lên đầu chuỗi


## 8. TRACE từng bước

In [7]:
def trace(s, k):
    L = lengths(s); print(f"s = {s!r}, độ dài sau mỗi bước: {L}, k = {k}")
    if k >= L[-1]: print("  k ngoài phạm vi -> '.'"); return
    for i in range(len(s) - 1, -1, -1):
        ch = s[i]; p = L[i]; old = k
        if ch == "#":
            if k >= p: k -= p
            print(f"  bước {i} '#': trước dài {p}; k {old} -> {k}")
        elif ch == "%":
            k = p - 1 - k; print(f"  bước {i} '%': trước dài {p}; k {old} -> {k}")
        elif ch == "*":
            print(f"  bước {i} '*': k giữ {k}")
        else:
            if k == p: print(f"  bước {i} '{ch}': k == {p} -> đáp án '{ch}'"); return
            print(f"  bước {i} '{ch}': k = {k} < {p}, bỏ qua")

trace("cd%#*#", 3)
trace("ab#c%#*x", 5)
print("brute:", brute("ab#c%#*x", 5))

s = 'cd%#*#', độ dài sau mỗi bước: [0, 1, 2, 2, 4, 3, 6], k = 3
  bước 5 '#': trước dài 3; k 3 -> 0
  bước 4 '*': k giữ 0
  bước 3 '#': trước dài 2; k 0 -> 0
  bước 2 '%': trước dài 2; k 0 -> 1
  bước 1 'd': k == 1 -> đáp án 'd'
s = 'ab#c%#*x', độ dài sau mỗi bước: [0, 1, 2, 4, 5, 5, 10, 9, 10], k = 5
  bước 7 'x': k = 5 < 9, bỏ qua
  bước 6 '*': k giữ 5
  bước 5 '#': trước dài 5; k 5 -> 0
  bước 4 '%': trước dài 5; k 0 -> 4
  bước 3 'c': k == 4 -> đáp án 'c'
brute: c


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def processStr(self, s: str, k: int) -> str:
        n = len(s)
        L = [0] * (n + 1)
        for i, ch in enumerate(s):
            m = L[i]
            if ch == '*':
                m = m - 1 if m else 0
            elif ch == '#':
                m <<= 1
            elif ch != '%':
                m += 1
            L[i + 1] = m
        if k >= L[n]:
            return '.'
        for i in range(n - 1, -1, -1):
            ch = s[i]
            p = L[i]
            if ch == '#':
                if k >= p:
                    k -= p
            elif ch == '%':
                k = p - 1 - k
            elif ch != '*':
                if k == p:
                    return ch
        return '.'

sol = Solution()
for s, k, exp in examples:
    got = sol.processStr(s, k)
    print(repr(s), k, "->", repr(got), "✔" if got == exp else "✘")

'a#b%*' 1 -> 'a' ✔
'cd%#*#' 3 -> 'd' ✔
'z*#' 0 -> '.' ✔


## 10. Vì sao thuật toán đúng

**Bất biến (đi ngược):** trước khi xử lý bước `i`, ký tự cần tìm bằng ký tự ở chỉ số `k` của chuỗi `R_{i+1}` (chuỗi sau bước `i`), với `0 ≤ k < L[i+1]`.

- Chữ `c`: `R_{i+1} = R_i + c`. Nếu `k = L[i]` thì ký tự là `c`; ngược lại `k < L[i]` và `R_{i+1}[k] = R_i[k]`.
- `*`: `R_{i+1}` là tiền tố của `R_i` ⇒ `R_{i+1}[k] = R_i[k]` và `k < L[i+1] ≤ L[i]`.
- `#`: `R_{i+1} = R_i + R_i` ⇒ `R_{i+1}[k] = R_i[k mod L[i]]`; vì `k < 2L[i]`, `k mod L[i] = k − L[i]` nếu `k ≥ L[i]`.
- `%`: `R_{i+1}[k] = R_i[L[i] − 1 − k]`.

Ban đầu bất biến đúng với `k < L[n]`. Ký tự có mặt trong chuỗi nên phải đến từ một bước thêm chữ nào đó ⇒ vòng lặp luôn trả về trước khi hết (dòng `return '.'` cuối chỉ là phòng hờ).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(độ dài kết quả)` — tới `10^{15}` | như trên |
| 2. Truy vết ngược | `O(n)` | `O(n)` |
| 3. Rope lười | `O(n)` dựng + `O(n)` mỗi truy vấn | `O(n)` |
| 4. Mô phỏng cắt ngắn | sai khi có `%` | — |

## 12. Bộ test

In [9]:
METHODS = [brute, reverse_track, rope, lambda s, k: Solution().processStr(s, k)]
for s, k, exp in examples:
    assert all(f(s, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [10]:
# 12.2 ca biên
assert Solution().processStr("*", 0) == "."
assert Solution().processStr("%%##", 0) == "."
assert Solution().processStr("a", 0) == "a" and Solution().processStr("a", 1) == "."
assert Solution().processStr("ab*%", 0) == "a"
print("Ca biên: OK")

Ca biên: OK


In [11]:
# 12.3 ngẫu nhiên (độ dài kết quả nhỏ): kiểm tra MỌI chỉ số k
rng = random.Random(3614)
for _ in range(1500):
    s = "".join(rng.choice("abc*#%") if rng.random() < 0.85 else "#" for _ in range(rng.randint(1, 12)))
    if lengths(s)[-1] > 5000: continue
    m = lengths(s)[-1]
    for k in range(m + 2):
        b = brute(s, k)
        assert all(f(s, k) == b for f in METHODS[1:]), (s, k)
print("1500 chuỗi ngẫu nhiên × mọi k: OK")

1500 chuỗi ngẫu nhiên × mọi k: OK


In [12]:
# 12.4 test lớn: kết quả dài ~10^15
rng = random.Random(1)
parts = []
for _ in range(10**5):
    r = rng.random()
    parts.append(rng.choice("abcdefghijklmnopqrstuvwxyz") if r < 0.6 else ("*" if r < 0.8 else "%"))
for pos in sorted(rng.sample(range(10**5), 30)): parts[pos] = "#"
s = "".join(parts); L = lengths(s)[-1]
t0 = time.time(); out = [Solution().processStr(s, k) for k in (0, L // 3, L - 1, L)]; t1 = time.time()
print("độ dài kết quả ~", f"{L:.3e}", "| ký tự:", out, "| thời gian 4 truy vấn:", round(t1 - t0, 2), "s")
assert L <= 10**15 and out == [rope(s, k) for k in (0, L // 3, L - 1, L)]

độ dài kết quả ~ 5.313e+12 | ký tự: ['l', 'z', 'e', '.'] | thời gian 4 truy vấn: 0.06 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cố dựng chuỗi (hết bộ nhớ)

In [13]:
print("chỉ cần 50 lần '#' sau một chữ cái: độ dài 2^50 =", 2 ** 50, "ký tự — không thể dựng")

chỉ cần 50 lần '#' sau một chữ cái: độ dài 2^50 = 1125899906842624 ký tự — không thể dựng


### Lỗi 2 — Với `*`, dùng `L − 1` không chặn ở 0

In [14]:
def wrong_pop_len(s, k):
    L = [0]
    for ch in s:
        m = L[-1]
        if ch == "*": m = m - 1                                       # SAI: có thể âm
        elif ch == "#": m *= 2
        elif ch != "%": m += 1
        L.append(m)
    if k >= L[-1]: return "."
    for i in range(len(s) - 1, -1, -1):
        ch = s[i]; p = L[i]
        if ch == "#":
            if k >= p: k -= p
        elif ch == "%": k = p - 1 - k
        elif ch != "*" and k == p: return ch
    return "."

print("sai:", repr(wrong_pop_len("*ab", 1)), "| đúng:", repr(brute("*ab", 1)))

sai: '.' | đúng: 'b'


### Lỗi 3 — Với `%` dùng độ dài **sau** sai lệch, hoặc quên `−1`

In [15]:
def wrong_reverse(s, k):
    L = lengths(s)
    if k >= L[-1]: return "."
    for i in range(len(s) - 1, -1, -1):
        ch = s[i]; p = L[i]
        if ch == "#":
            if k >= p: k -= p
        elif ch == "%": k = p - k                                       # SAI: thiếu -1
        elif ch != "*" and k == p: return ch
    return "."

print("sai:", repr(wrong_reverse("ab%", 0)), "| đúng:", repr(brute("ab%", 0)))

sai: '.' | đúng: 'b'


### Lỗi 4 — Kiểm tra chữ cái bằng `k == L[i+1]` (độ dài **sau**)

In [16]:
def wrong_letter(s, k):
    L = lengths(s)
    if k >= L[-1]: return "."
    for i in range(len(s) - 1, -1, -1):
        ch = s[i]; p = L[i]
        if ch == "#":
            if k >= p: k -= p
        elif ch == "%": k = p - 1 - k
        elif ch != "*" and k == L[i + 1]: return ch                    # SAI
    return "."

print("sai:", repr(wrong_letter("ab", 1)), "| đúng:", repr(brute("ab", 1)))

sai: 'a' | đúng: 'b'


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi/dãy được tạo bằng các thao tác làm nó dài theo cấp số nhân, hỏi một vị trí" ⇒ lưu **độ dài**, đi **ngược** để ánh xạ chỉ số (LC 880 *Decoded String at Index*, LC 3304/3307 *Find the K-th Character*).
- Mỗi thao tác chỉ cần một **ánh xạ ngược** `k_sau → k_trước`.
- Thao tác không khả nghịch về độ dài (như `max(0, L−1)`) ⇒ lưu mảng độ dài thay vì tính ngược.
- Khi kiểm thử, so **mọi** chỉ số `k` (kể cả `k = độ dài` và lớn hơn) với mô phỏng trên chuỗi nhỏ — lỗi lệch 1 ở `%` hay ở bước thêm chữ lộ ra ngay.
- Nếu đề cho cả thao tác "chèn đầu" hoặc "xoá đầu", ánh xạ ngược chỉ là cộng/trừ 1 vào `k` — khung không đổi.

## 15. Biến thể của bài

1. **Bản I** (`|s| ≤ 20`): mô phỏng trực tiếp.
2. **LC 880**: chữ số `d` lặp chuỗi `d` lần ⇒ `k ← k mod L_trước`.
3. **Nhiều truy vấn `k`**: dùng chung mảng độ dài, mỗi truy vấn `O(n)` — hoặc nén các bước "giữ nguyên" để đi nhanh hơn.
4. **Thêm thao tác `@`: xoá ký tự đầu**: ánh xạ ngược `k ← k + 1`.

In [17]:
# Biến thể 2 + 4: chữ số d lặp d lần, '@' xoá ký tự đầu
def generalized(s, k):
    L = [0]
    for ch in s:
        m = L[-1]
        if ch == "*": m = max(0, m - 1)
        elif ch == "@": m = max(0, m - 1)
        elif ch == "#": m *= 2
        elif ch == "%": pass
        elif ch.isdigit(): m *= int(ch)
        else: m += 1
        L.append(m)
    if k >= L[-1]: return "."
    for i in range(len(s) - 1, -1, -1):
        ch = s[i]; p = L[i]
        if ch == "#":
            if k >= p: k -= p
        elif ch.isdigit(): k %= p
        elif ch == "%": k = p - 1 - k
        elif ch == "@": k += 1
        elif ch == "*": pass
        elif k == p: return ch
    return "."

def gen_brute(s, k):
    r = []
    for ch in s:
        if ch == "*":
            if r: r.pop()
        elif ch == "@":
            if r: r.pop(0)
        elif ch == "#": r = r + r
        elif ch == "%": r.reverse()
        elif ch.isdigit(): r = r * int(ch)
        else: r.append(ch)
    return r[k] if 0 <= k < len(r) else "."

rng = random.Random(15)
for _ in range(1000):
    s = "".join(rng.choice("ab*#%@23") for _ in range(rng.randint(1, 10)))
    for k in range(25): assert generalized(s, k) == gen_brute(s, k), (s, k)
print("Biến thể 2 + 4 OK")

Biến thể 2 + 4 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Mô phỏng | Dựng chuỗi | `O(|result|)` | Bản I |
| 2. Truy vết ngược | Ánh xạ chỉ số | `O(n)` | **Lời giải chuẩn** |
| 3. Rope lười | Cây thao tác | `O(n)` | Nhiều truy vấn, thao tác phức tạp |
| 4. Cắt ngắn | Giữ tiền tố | sai với `%` | Minh hoạ lỗi |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `processStr("ab#", 3)`.

**Bài 2 (tính tay).** Trace ngược cho `s = "xy%#z", k = 2`.

**Bài 3 (điền TODO).** Viết `final_length(s)`.

**Bài 4 (tự viết lại).** Viết `many_queries(s, ks)` trả lời nhiều `k` dùng chung mảng độ dài.

**Bài 5 (mini challenge).** Trả về **đoạn con** `result[a..b]` (độ dài ≤ 100) mà không dựng chuỗi.

In [18]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def final_length(s):
    # TODO
    pass

In [19]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def substring(s, a, b):
    # TODO: gọi truy vết ngược cho từng vị trí
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `"ab" → "abab"`. (2) Chỉ số 3 là `b`. (3) `"b"`.

**Bài 2.** (1) Độ dài `[0,1,2,2,4,5]`. (2) Ngược: `z` (p=4): `k=2 < 4`; `#` (p=2): `k=2 ≥ 2 ⇒ 0`; `%` (p=2): `k = 1`. (3) `y` (p=1): `k == 1` ⇒ `"y"`.

**Bài 3.** (1) Như `lengths`. (2) Trả phần tử cuối. (3) Kiểm chứng mô phỏng.

**Bài 4.** (1) Tính `L` một lần. (2) Với mỗi `k` chạy vòng ngược. (3) `O(n · |ks|)`.

**Bài 5.** (1) Mỗi vị trí trong `[a, b]` là một truy vấn. (2) Ghép kết quả. (3) Nếu vượt độ dài thì dừng.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
assert Solution().processStr("ab#", 3) == "b"
assert Solution().processStr("xy%#z", 2) == "y" == brute("xy%#z", 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [21]:
# Bài 3, 4
def final_length_sol(s): return lengths(s)[-1]

def many_queries_sol(s, ks):
    L = lengths(s); out = []
    for k0 in ks:
        k = k0
        if k >= L[-1]: out.append("."); continue
        for i in range(len(s) - 1, -1, -1):
            ch = s[i]; p = L[i]
            if ch == "#":
                if k >= p: k -= p
            elif ch == "%": k = p - 1 - k
            elif ch != "*" and k == p: out.append(ch); break
    return out

rng = random.Random(4)
for _ in range(500):
    s = "".join(rng.choice("abc*#%") for _ in range(rng.randint(1, 10)))
    r = []
    for ch in s:
        if ch == "*":
            if r: r.pop()
        elif ch == "#": r = r + r
        elif ch == "%": r.reverse()
        else: r.append(ch)
    assert final_length_sol(s) == len(r)
    ks = list(range(len(r) + 2)); assert many_queries_sol(s, ks) == [brute(s, k) for k in ks]
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [22]:
# Bài 5
def substring_sol(s, a, b):
    return "".join(c for c in many_queries_sol(s, list(range(a, b + 1))) if c != ".")

print("Bài 5:", substring_sol("cd%#*#", 1, 4), "| brute:", "".join(brute("cd%#*#", k) for k in range(1, 5)))
assert substring_sol("cd%#*#", 1, 4) == "cddc"

Bài 5: cddc | brute: cddc


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
L[i+1] = L[i] + 1 (chữ) | max(0, L[i]-1) (*) | 2*L[i] (#) | L[i] (%)
if k >= L[n]: return '.'
for i = n-1 .. 0, p = L[i]:
    '#': if k >= p: k -= p
    '%': k = p - 1 - k
    '*': giữ
    chữ: if k == p: return chữ
```

**Checklist:**

- [ ] Không dựng chuỗi?
- [ ] `*` chặn độ dài ở 0?
- [ ] `%` dùng `p − 1 − k` với độ dài **trước**?
- [ ] Chữ cái so `k == L[i]` (độ dài trước)?
- [ ] `k ≥ độ dài cuối` ⇒ `'.'`?